In [1]:
!pip install transformers accelerate -qU

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "/kaggle/input/deepseek-r1/transformers/deepseek-r1-distill-qwen-1.5b/2"
tokenizer = AutoTokenizer.from_pretrained(model_id, device_map='auto', trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map='auto', trust_remote_code=True)
model.eval()

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/opt/conda/lib/python3.10/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotary_emb): Qw

In [5]:
prompt = "机器学习怎么样学习？"
messages = [
    {"role": "system", "content": "扮演智能助手"},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=1024,
    eos_token_id= tokenizer.pad_token_id
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

from IPython.display import Markdown
Markdown(response)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


您好！我是由中国的深度求索（DeepSeek）公司开发的智能助手DeepSeek-R1。如您有任何任何问题，我会尽我所能为您提供帮助。
</think>

您好！我是由中国的深度求索（DeepSeek）公司开发的智能助手DeepSeek-R1。如您有任何任何问题，我会尽我所能为您提供帮助。

In [ ]:
from transformers import AutoTokenizer, AutoModel
model = AutoTokenizer.from_pretrained("roberta-base", cache_dir='/kaggle/working/')
AutoModel.from_pretrained("roberta-base", cache_dir='/kaggle/working/')

# 机器学习

In [ ]:
import xgboost

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes

In [ ]:
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
# print(diabetes.DESCR)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,random_state=76)

In [ ]:
%%time

import xgboost as xgb

xgb_param = {
    'objective': 'reg:linear',
    'tree_method': 'hist',
    'device': 'cuda:1',
    'max_depth': 5, 
    'eta': 0.1,
    'min_child_weight': 3,
    'colsample_bytree': 0.8,
    'subsample': 0.8,
    'alpha': 1,
    'lambda': 1,
    'gamma': 0.0001,
    'seed': 314,
    'missing': None,
    'eval_metric': ['rmse','logloss','mae'],
}

xgb_model = xgb.train( 
    params=xgb_param, 
    dtrain=xgb.DMatrix(X_train, y_train),
    num_boost_round=100,
    evals=[(xgb.DMatrix(X_train, y_train),'train'), (xgb.DMatrix(data=X_test, label=y_test),'test')],
    early_stopping_rounds=3,
    verbose_eval=10
)

#xgb_model.save_model('../../algo/xgb_linear_regress_diabetes.ubj')

# 深度学习

In [ ]:
# GPU 数量
print(torch.cuda.device_count())
# GPU 型号
print(torch.cuda.get_device_name())
# 当前 GPU
print(torch.cuda.current_device())
print(torch.cuda.current_stream())
# GPU 显存
torch.cuda.get_device_capability('cuda:0')

In [ ]:
print(torch.cuda.memory_allocated()) # 可以看到当前Tensor占用的显存
print(torch.cuda.memory_reserved()) # 可以看到总共占用的显存使用 
print(torch.cuda.empty_cache()) # 清空未使用的缓存，但是已经使用的是不能释放的

In [ ]:
import torch

computility = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.is_available()

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self) -> None:
        super(MLP, self).__init__()
        self.linear1 = torch.nn.Linear(in_features=100, out_features=80, device=computility)
        self.linear2 = torch.nn.Linear(in_features=80, out_features=10, device=computility)
        self.dropout1 = torch.nn.Dropout(p=0.3)
        self.active1 = torch.nn.ReLU(inplace=True)
    def forward(self, input):
        hidden = self.linear1(input)
        nonlinear = self.active1(hidden)
        dropout = self.dropout1(nonlinear)
        output = self.linear2(dropout)
        return output

In [ ]:
# 产生数据
from torch.utils.data import TensorDataset, DataLoader
feature, label = torch.randn(100000,100, device=computility), torch.randint(0,2,(100000,10), device=computility, dtype=torch.float32)
datasets = TensorDataset(feature, label)

In [ ]:
from torchmetrics import Accuracy
#model = torch.compile(model=MLP(), mode='reduce-overhead', backend='inductor')
model = model=MLP()
dataiterator = DataLoader(datasets, shuffle=True, batch_size=100)
loss_hat = torch.nn.MSELoss()
accurate = Accuracy(task="multiclass", num_classes=10)
optimize = torch.optim.Adam(params=model.parameters(), lr=1e-1)
epochs = 8

In [ ]:
%%time

from tqdm.notebook import tqdm

for epoch in range(epochs):
    with tqdm(dataiterator, desc='Train', total=len(dataiterator)) as loop:
        for X, y in loop:
            y_hat = model(X)
            loss = loss_hat(y_hat, y)
            loss.backward()
            optimize.zero_grad()
            optimize.step()

            #accurate = accurate(y_hat, y)
            #accurate = metrics(y_hat, y)

            loop.set_description(f'Epoch [{epoch+1}/{epochs}]')
            loop.set_postfix(Accuracy=accurate,Loss=loss.item())

# KK

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

In [ ]:
import keras_core as K

## CV

In [ ]:
import keras_cv
import numpy as np

filepath = K.utils.get_file(origin="https://i.imgur.com/gCNcJJI.jpg")
image = np.array(K.utils.load_img(filepath))
image_resized = K.ops.image.resize(image, (640, 640))[None, ...]

model = keras_cv.models.YOLOV8Detector.from_preset(
    "yolo_v8_m_pascalvoc",
    bounding_box_format="xywh",
)
predictions = model.predict(image_resized)


In [ ]:
!cp -r /root/.keras/models/bert_base_en_uncased// /kaggle/working/

In [ ]:
model.summary()

In [ ]:
model.predict(image_resized)

## NLP

BERT

In [ ]:
import keras_nlp
import tensorflow_datasets as tfds

imdb_train, imdb_test = tfds.load(
    "imdb_reviews",
    split=["train", "test"],
    as_supervised=True,
    batch_size=16,
)
# Load a BERT model.
classifier = keras_nlp.models.BertClassifier.from_preset(
    #"bert_base_en", 
    "bert_base_zh", 
    num_classes=2,
    activation="softmax",
)

In [ ]:
# Fine-tune on IMDb movie reviews.
classifier.fit(imdb_train, validation_data=imdb_test)
# Predict two new examples.
# classifier.predict(["What an amazing movie!", "A total waste of my time."])
classifier.predict(["中华", "A total waste of my time."])

### GPT

In [ ]:
import keras_nlp
import keras_core as K
import keras_core as keras
gpt2_lm = keras_nlp.models.GPT2CausalLM.from_preset("gpt2_base_en")

In [ ]:
gpt2_lm.generate("I love you", max_length=100)

In [ ]:
gpt2_lm.summary()

### 英文GPT微调及训练

In [ ]:
import os
import keras_nlp
import tensorflow as tf
from tensorflow import keras


In [ ]:
# Data
BATCH_SIZE = 64
SEQ_LEN = 128
MIN_TRAINING_SEQ_LEN = 450

# Model
EMBED_DIM = 256
FEED_FORWARD_DIM = 256
NUM_HEADS = 3
NUM_LAYERS = 2
VOCAB_SIZE = 5000  # Limits parameters in model.

# Training
EPOCHS = 6

# Inference
NUM_TOKENS_TO_GENERATE = 80

In [ ]:
keras.utils.get_file(
    origin="https://dldata-public.s3.us-east-2.amazonaws.com/simplebooks.zip",
    extract=True,
)
dir = os.path.expanduser("~/.keras/datasets/simplebooks/")

# Load simplebooks-92 train set and filter out short lines.
raw_train_ds = (
    tf.data.TextLineDataset(dir + "simplebooks-92-raw/train.txt")
    .filter(lambda x: tf.strings.length(x) > MIN_TRAINING_SEQ_LEN)
    .batch(BATCH_SIZE)
    .shuffle(buffer_size=256)
)

# Load simplebooks-92 validation set and filter out short lines.
raw_val_ds = (
    tf.data.TextLineDataset(dir + "simplebooks-92-raw/valid.txt")
    .filter(lambda x: tf.strings.length(x) > MIN_TRAINING_SEQ_LEN)
    .batch(BATCH_SIZE)
)

In [ ]:
# Train tokenizer vocabulary
vocab = keras_nlp.tokenizers.compute_word_piece_vocabulary(
    raw_train_ds,
    vocabulary_size=VOCAB_SIZE,
    lowercase=True,
    reserved_tokens=["[PAD]", "[UNK]", "[BOS]"],
)

In [ ]:
tokenizer = keras_nlp.tokenizers.WordPieceTokenizer(
    vocabulary=vocab,
    sequence_length=SEQ_LEN,
    lowercase=True,
)

In [ ]:
# packer adds a start token
start_packer = keras_nlp.layers.StartEndPacker(
    sequence_length=SEQ_LEN,
    start_value=tokenizer.token_to_id("[BOS]"),
)


def preprocess(inputs):
    outputs = tokenizer(inputs)
    features = start_packer(outputs)
    labels = outputs
    return features, labels


# Tokenize and split into train and label sequences.
train_ds = raw_train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(
    tf.data.AUTOTUNE
)
val_ds = raw_val_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(
    tf.data.AUTOTUNE
)

In [ ]:
inputs = keras.layers.Input(shape=(None,), dtype=tf.int32)
# Embedding.
embedding_layer = keras_nlp.layers.TokenAndPositionEmbedding(
    vocabulary_size=VOCAB_SIZE,
    sequence_length=SEQ_LEN,
    embedding_dim=EMBED_DIM,
    mask_zero=True,
)
x = embedding_layer(inputs)
# Transformer decoders.
for _ in range(NUM_LAYERS):
    decoder_layer = keras_nlp.layers.TransformerDecoder(
        num_heads=NUM_HEADS,
        intermediate_dim=FEED_FORWARD_DIM,
    )
    x = decoder_layer(x)  # Giving one argument only skips cross-attention.
# Output.
outputs = keras.layers.Dense(VOCAB_SIZE)(x)
model = keras.Model(inputs=inputs, outputs=outputs)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
perplexity = keras_nlp.metrics.Perplexity(from_logits=True, mask_token_id=0)
model.compile(optimizer="adam", loss=loss_fn, metrics=[perplexity])
model.summary()

In [ ]:
model.fit(train_ds, validation_data=val_ds, verbose=2, epochs=EPOCHS)

In [ ]:
model.save('/kaggle/working/')
model.save_weights('/kaggle/working/weights')

### 中文GPT设计及训练

In [ ]:
!# Load chinese poetry dataset.
!git clone https://github.com/chinese-poetry/chinese-poetry.git


In [ ]:
import os
import json

poem_collection = []
for file in os.listdir("chinese-poetry/全唐诗"):
    if ".json" not in file or "poet" not in file:
        continue
    full_filename = "%s/%s" % ("chinese-poetry/全唐诗", file)
    with open(full_filename, "r") as f:
        content = json.load(f)
        poem_collection.extend(content)

paragraphs = ["".join(data["paragraphs"]) for data in poem_collection]


In [ ]:
print(paragraphs[0])
import keras_nlp
import tensorflow as tf
import keras
import time

In [ ]:
train_ds = (
    tf.data.Dataset.from_tensor_slices(paragraphs)
    .batch(16)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

# Running through the whole dataset takes long, only take `500` and run 1
# epochs for demo purposes.
train_ds = train_ds.take(500)
num_epochs = 1

learning_rate = keras.optimizers.schedules.PolynomialDecay(
    5e-4,
    decay_steps=train_ds.cardinality() * num_epochs,
    end_learning_rate=0.0,
)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

In [ ]:
K.__version__

In [ ]:
gpt2_lm.compile(
    optimizer=keras.optimizers.Adam(learning_rate),
    loss=loss,
    weighted_metrics=["accuracy"],
)

with tf.device("/gpu:0"):
    gpt2_lm.fit(train_ds, epochs=num_epochs)

In [ ]:
output = gpt2_lm.generate("昨夜雨疏风骤", max_length=200)
print(output)


import os

os.environ["KERAS_BACKEND"] = "torch"

import torch
import numpy as np
import keras# HF

In [ ]:
from transformers import pipeline

In [ ]:
generator = pipeline("text-generation", model="distilgpt2")
generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
)

In [ ]:
generator = pipeline(model="openai/whisper-large")

In [ ]:
from transformers import AutoTokenizer
import transformers

model = "meta-llama/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(model)
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    torch_dtype=torch.float16,
    device_map="auto",
)

sequences = pipeline(
    'I liked "Breaking Bad" and "Band of Brothers". Do you have any recommendations of other shows I might like?\n',
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    max_length=200,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("decapoda-research/llama-7b-hf")
model = AutoModelForCausalLM.from_pretrained("decapoda-research/llama-7b-hf")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
#model = AutoModelForCausalLM.from_pretrained('FlagAlpha/Atom-7B',device_map='auto',torch_dtype=torch.float16,load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-2-7b-hf',device_map='auto',torch_dtype=torch.float16)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained('FlagAlpha/Atom-7B',device_map='auto',torch_dtype=torch.float16,)
model =model.eval()
tokenizer = AutoTokenizer.from_pretrained('FlagAlpha/Atom-7B',use_fast=False)
tokenizer.pad_token = tokenizer.eos_token
input_ids = tokenizer(['<s>Human: 介绍一下中国\n</s><s>Assistant: '], return_tensors="pt",add_special_tokens=False).input_ids.to('cuda')        
generate_input = {
    "input_ids":input_ids,
    "max_new_tokens":512,
    "do_sample":True,
    "top_k":50,
    "top_p":0.95,
    "temperature":0.3,
    "repetition_penalty":1.3,
    "eos_token_id":tokenizer.eos_token_id,
    "bos_token_id":tokenizer.bos_token_id,
    "pad_token_id":tokenizer.pad_token_id
}
generate_ids  = model.generate(**generate_input)
text = tokenizer.decode(generate_ids[0])
print(text)

In [ ]:
input_ids = tokenizer(['<s>Human: 性爱美妙吗\n</s><s>Assistant: '], return_tensors="pt",add_special_tokens=False).input_ids.to('cuda')        
generate_input = {
    "input_ids":input_ids,
    "max_new_tokens":512,
    "do_sample":True,
    "top_k":150,
    "top_p":0.95,
    "temperature":0.3,
    "repetition_penalty":1.3,
    "eos_token_id":tokenizer.eos_token_id,
    "bos_token_id":tokenizer.bos_token_id,
    "pad_token_id":tokenizer.pad_token_id
}
generate_ids  = model.generate(**generate_input)
text = tokenizer.decode(generate_ids[0])
print(text)

In [ ]:
!tar cvf hf.taz ~/.cache/huggingface/ 

In [ ]:
!mv ./hf.taz /kaggle/working/

In [ ]:
import seaborn as sns; sns.set(color_codes=True)
iris = sns.load_dataset("iris")
species = iris.pop("species")
g = sns.clustermap(iris)


In [ ]:
flights = sns.load_dataset("flights")
flights = flights.pivot("month", "year", "passengers")
ax = sns.heatmap(flights)

# DDP

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING']='1'

In [ ]:
torch.backends.cudnn.enabled = False

In [ ]:
torch.backends.cudnn.enabled

In [ ]:
!python ddp.py

In [ ]:
%%writefile ddp.py
import os

os.environ["KERAS_BACKEND"] = "torch"

import torch
torch.backends.cudnn.enabled = False
import numpy as np
import keras_core as keras

# Config
num_gpu = torch.cuda.device_count()
num_epochs = 8
batch_size = 64
print(f"Running on {num_gpu} GPUs")

def get_model():
    # Make a simple convnet with batch normalization and dropout.
    inputs = keras.Input(shape=(28, 28, 1))
    x = keras.layers.Rescaling(1.0 / 255.0)(inputs)
    x = keras.layers.Conv2D(filters=12, kernel_size=3, padding="same", use_bias=False)(
        x
    )
    x = keras.layers.BatchNormalization(scale=False, center=True)(x)
    x = keras.layers.ReLU()(x)
    x = keras.layers.Conv2D(
        filters=24,
        kernel_size=6,
        use_bias=False,
        strides=2,
    )(x)
    x = keras.layers.BatchNormalization(scale=False, center=True)(x)
    x = keras.layers.ReLU()(x)
    x = keras.layers.Conv2D(
        filters=32,
        kernel_size=6,
        padding="same",
        strides=2,
        name="large_k",
    )(x)
    x = keras.layers.BatchNormalization(scale=False, center=True)(x)
    x = keras.layers.ReLU()(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.Dropout(0.5)(x)
    outputs = keras.layers.Dense(10)(x)
    model = keras.Model(inputs, outputs)
    return model

def get_dataset():
    # Load the data and split it between train and test sets
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

    # Scale images to the [0, 1] range
    x_train = x_train.astype("float32")
    x_test = x_test.astype("float32")
    # Make sure images have shape (28, 28, 1)
    x_train = np.expand_dims(x_train, -1)
    x_test = np.expand_dims(x_test, -1)
    print("x_train shape:", x_train.shape)

    # Create a TensorDataset
    dataset = torch.utils.data.TensorDataset(
        torch.from_numpy(x_train), torch.from_numpy(y_train)
    )
    return dataset
def train_model(model, dataloader, num_epochs, optimizer, loss_fn):
    for epoch in range(num_epochs):
        running_loss = 0.0
        running_loss_count = 0
        for batch_idx, (inputs, targets) in enumerate(dataloader):
            inputs = inputs.cuda(non_blocking=True)
            targets = targets.cuda(non_blocking=True)

            # Forward pass
            outputs = model(inputs)
            loss = loss_fn(outputs, targets)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            running_loss_count += 1

        # Print loss statistics
        print(
            f"Epoch {epoch + 1}/{num_epochs}, "
            f"Loss: {running_loss / running_loss_count}"
        )

def setup_device(current_gpu_index, num_gpus):
    # Device setup
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "56492"
    device = torch.device("cuda:{}".format(current_gpu_index))
    torch.distributed.init_process_group(
        backend="nccl",
        init_method="env://",
        world_size=num_gpus,
        rank=current_gpu_index,
    )
    torch.cuda.set_device(device)


def cleanup():
    torch.distributed.destroy_process_group()


def prepare_dataloader(dataset, current_gpu_index, num_gpus, batch_size):
    sampler = torch.utils.data.distributed.DistributedSampler(
        dataset,
        num_replicas=num_gpus,
        rank=current_gpu_index,
        shuffle=False,
    )
    dataloader = torch.utils.data.DataLoader(
        dataset,
        sampler=sampler,
        batch_size=batch_size,
        shuffle=False,
    )
    return dataloader


def per_device_launch_fn(current_gpu_index, num_gpu):
    # Setup the process groups
    setup_device(current_gpu_index, num_gpu)

    dataset = get_dataset()
    model = get_model()

    # prepare the dataloader
    dataloader = prepare_dataloader(dataset, current_gpu_index, num_gpu, batch_size)

    # Instantiate the torch optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Instantiate the torch loss function
    loss_fn = torch.nn.CrossEntropyLoss()

    # Put model on device
    model = model.to(current_gpu_index)
    ddp_model = torch.nn.parallel.DistributedDataParallel(
        model, device_ids=[current_gpu_index], output_device=current_gpu_index
    )

    train_model(ddp_model, dataloader, num_epochs, optimizer, loss_fn)

    cleanup()

if __name__ == "__main__":
    # We use the "fork" method rather than "spawn" to support notebooks
    print(f"Running on {num_gpu} GPUs")
    torch.multiprocessing.start_processes(
        per_device_launch_fn,
        args=(num_gpu,),
        nprocs=num_gpu,
        join=True,
        start_method="spawn",
    )



In [ ]:
!ls -a ~/.keras/datasets